# Exp9.0 — Rotating within-user vs cross-user CV

Aggregation-only notebook for protocol `rotating_grouped_cv_v2`. The primary result is pooled out-of-fold (OOF) test performance.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'scripts').exists() and (path / 'notebooks').exists():
            return path
    raise RuntimeError('Repository root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_9_0_within_user_generalization' / 'rotating_grouped_cv_v2'
audit = json.loads((root / 'audit.json').read_text())
audit


## Fold construction

- `within_user`: segment-level stratified folds inside each user.
- `cross_user`: user-level folds; test users are unseen during training.
- Each rotation uses 3 folds train, 1 validation, 1 test.

In [ ]:
fold_summary = pd.read_csv(root / 'fold_summary.csv')
rotation_summary = pd.read_csv(root / 'rotation_summary.csv')
display(fold_summary)
display(rotation_summary)


## Primary OOF results

Every sample is held out as test exactly once per CV mode and method. These pooled metrics are the primary comparison.

In [ ]:
oof = pd.read_csv(root / 'oof_test_metrics.csv')
gap = pd.read_csv(root / 'within_vs_cross_user.csv')
display(oof.sort_values(['method', 'cv_mode']))
display(gap)


## Fold-level stability

Use fold mean/std as a variance diagnostic, not as the primary pooled estimate.

In [ ]:
metric_summary = pd.read_csv(root / 'metric_summary.csv')
metric_runs = pd.read_csv(root / 'metric_runs.csv')
display(metric_summary)
display(metric_runs[metric_runs['split'].eq('test')].sort_values(['cv_mode','method','rotation']))


## Per-user and per-class OOF diagnostics

In [ ]:
per_user = pd.read_csv(root / 'oof_per_user_test.csv')
per_class = pd.read_csv(root / 'oof_per_class_test.csv')
display(per_user.sort_values(['cv_mode','method','balanced_accuracy']))
display(per_class.sort_values(['cv_mode','method','class_label']))


## Pooled OOF confusion matrices

In [ ]:
conf = pd.read_csv(root / 'confusion_summary.csv')
for (cv_mode, method), frame in conf.groupby(['cv_mode','method'], sort=False):
    print(f'\n{cv_mode} / {method}')
    display(frame.pivot(index='true_label', columns='pred_label', values='count'))
